In [ ]:
import os
os.environ["VLLM_CONFIGURE_LOGGING"] = "0"
import logging
logging.basicConfig(format='%(message)s', level=logging.FATAL+1)

import sys
sys.path.append("..")

import time

import random
import numpy as np

import torch
from vllm import LLM, SamplingParams

from sal.config import Config

from core import bon_search_v1

from utils.load_data import load_data_hf

In [ ]:
# base_dir
base_dir = '/groups/chichengz/tnn/datasets/'

# dataset path
ds_name = "prm800k"
ds_split = "test"
ds_dir = base_dir + "/prm800k/math_splits"

# llm and prm path
# llm_dir = base_dir + "/Llama-3.2-1B-Instruct-GGUF/Llama-3.2-1B-Instruct.Q4_K_M.gguf"
# prm_dir = base_dir + "/Llama3.1-8B-PRM-Deepseek-Data-GGUF/Llama3.1-8B-PRM-Deepseek-Data.Q4_K_M.gguf"

llm_dir = base_dir + "Llama3.2-1B-Instruct"
prm_dir = base_dir + "Llama3.1-8B-PRM-Deepseek-Data"

In [ ]:
# general params
config = Config()
config.agg_strategy = 'last'
config.temperature = 0.8
config.max_tokens = 2048

config.n = 256
config.filter_duplicates = True
config.date_string = "Aug 1 2025"
config.seed = 0

llm_gpu_memory_utilization = 0.7
llm_vllm = LLM(
    model=llm_dir,
    tensor_parallel_size=1,
    max_model_len=5000,
    gpu_memory_utilization=llm_gpu_memory_utilization,
    enforce_eager=True,
    distributed_executor_backend=None,
    dtype="float16",
    seed=config.seed,
)

In [ ]:
num_trials = 2
level = 4
dataset = load_data_hf(ds_dir, ds_split=ds_split, level=level)

num_questions = len(dataset)
batch_of_questions = [dataset[i]['question'] for i in range(num_questions)]

1: 43
2: 90
3: 105
4: 128
5: 134


### Best-of-N Variants
- **v1 (native)**: single prompt, `n=config.n` — vLLM generates all completions internally
- **v2 (prompt_dup)**: duplicate each prompt `config.n` times, `n=1` — continuous batching across duplicated inputs

In [ ]:

for method_name, test_method in [
    ('native     (v1)', bon_search_v1.best_of_n_v1),
    ('prompt_dup (v2)', bon_search_v1.best_of_n_v2),
]:
    print(f"\n--- {method_name} ---")
    start_time = time.time()
    for trial_idx in range(num_trials):
        print(f"trial {trial_idx}")
        test_method(batch_of_questions, config, llm_vllm, trial_idx)
        total_time = time.time() - start_time
        time_per_trial = total_time / (trial_idx + 1)
        time_per_question = time_per_trial / num_questions
        print(f"  {time_per_question:.4f}s per question")
        print(f"  {time_per_trial:.4f}s per trial")
    print(f"  {total_time:.4f}s total")